# D2.3 · Scoping an agentic incident

**Function D — The Agentic SOC → The Agentic SOC — Response**  ·  *Security of AI*

Builds on **[D2.2 · When the actor is an agent](https://spbreed.github.io/cyber-commons/lessons/D2.2.html)**.

| | |
|---|---|
| Tools used | OpenTelemetry, Kimi K2, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Scope a multi-agent incident end to end.

**Why a security engineer needs it.** The initiating agent is not the acting one. The control it builds is: reconstruct the action chain across all three planes.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

The agent acted for eleven minutes on delegated credentials at machine speed. Scoping that means reconstructing blast radius from identity and egress logs, because asking what it touched is not a question anyone can answer from memory.

> **At CyberTravels.** Eleven minutes of CyberTravels on delegated credentials. What it touched is not answerable from memory — it comes out of the identity and egress logs, if they exist. R9.

## 2 · The framework

```
   11 minutes at machine speed

   identity log ---+                    +--> resources touched
                   +--> reconstruct --> +--> data read
   egress log   ---+                    +--> destinations reached
                                        +--> credentials used

   the question "what did it touch" is not answerable from memory
```

Scoping answers "what was touched?" For a host-based incident you enumerate
hosts. For an agentic incident, **scope follows the delegation graph**.

The agent that touched the resource is usually the *last* actor in a chain. If
you scope only that actor, you miss everything the earlier actors reached — and
because authority narrows down the chain, the earlier actors typically had
*more* access, not less.

The undercount is systematic and it grows with delegation depth, which is the
operational reason B2.0 bounds delegation depth in the first place.

## 3 · Scoping as a skill

Scoping a human incident asks where someone logged in. Scoping this one asks what the agent **decided** — every action was individually authorised, so nothing looks wrong at the authentication layer.

Two fields in the contract carry most of the weight. `reach` and `confirmed_exfiltration` are separate numbers, because reach is the scope until proven otherwise and the smaller number must never stand in for the larger in a notification decision. And `does_not_stop` makes containment state its own limits.

In [ ]:
# skills/secops/incident-scoping/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: incident-scoping
description: >-
  Scope an incident in which an agent was the actor — what it touched, what it
  changed, what it may have exfiltrated, and where containment must cut. Use
  when responding to an agent-involved incident, reconstructing what an
  autonomous system did, deciding what to revoke, or sizing notification
  obligations.
allowed-tools: Read, Grep, Bash
---

# Scoping an agentic incident

Scoping a human incident asks where someone logged in. Scoping an agentic one
asks what the agent **decided**, because a compromised agent's actions are all
individually authorised. Nothing looks anomalous at the authentication layer;
the anomaly is in the sequence.

## When to use this

Any incident where an agent, an automated pipeline, or an AI-driven tool
performed actions under investigation.

## Procedure

**1 — Fix the window.** Establish the first suspicious decision, not the first
alert. Work backwards from the earliest action you cannot explain; the trigger
is usually earlier than the detection by the length of one task loop.

**2 — Reconstruct the decision chain.** For the window, list every action with
the input that motivated it. The critical question is which input entered the
context from **outside the trust boundary** — a fetched page, an issue comment,
a dependency's README, a tool description. That input is the likely root cause,
and it is invisible if you only log tool calls and not their justification.

**3 — Separate authority from behaviour.** For each action ask: was it within
the agent's granted authority? Actions that were authorised but wrong tell you
the grant was too broad. Actions that exceeded authority tell you a control
failed. These lead to different fixes and must not be pooled.

**4 — Establish data reach.** What did the agent read, and where could it have
sent it? Reach is bounded by the agent's egress, not by what it appears to have
sent — a request body you cannot see is still reach. State reach and confirmed
exfiltration as separate numbers, and never let the smaller one stand in for
the larger in a notification decision.

**5 — Decide the containment cut.** Options, in increasing cost: revoke the
agent's credential, disable the trigger, quarantine the workload, disable the
whole class of agents. Choose by blast radius, not by convenience, and record
what the cut does **not** stop — sibling agents on the same shared service
account almost always survive a credential revocation aimed at one of them.

**6 — Preserve evidence the agent could alter.** If the agent can write to the
log store, the logs are not evidence. Snapshot first, then contain.

## Output contract

```json
{
  "window": {"first_suspicious_action": "str", "detected_at": "str", "gap_seconds": 0},
  "chain": [{"action": "str", "motivating_input": "str",
             "input_origin": "operator|internal|external_untrusted",
             "within_authority": true}],
  "root_cause": {"input": "str", "origin": "str", "why_trusted": "str"},
  "authority": {"authorised_but_wrong": 0, "exceeded_authority": 0},
  "data": {"reach": ["str"], "confirmed_exfiltration": ["str"], "egress_bounded_by": "str"},
  "containment": {"cut": "credential|trigger|workload|class",
                  "does_not_stop": ["str"], "evidence_snapshotted_first": true},
  "clock": {"regulatory_trigger": false, "basis": "str"}
}
```

## Failure modes

- **Scoping by authentication.** Every action was authenticated; that is the
  point.
- **Logging tool calls without their motivating input.** Root cause then cannot
  be established at all.
- **Reporting confirmed exfiltration as the scope.** Reach is the scope until
  proven otherwise.
- **Revoking one agent's token** when the identity is shared, and calling it
  contained.
- **Containing before snapshotting** a log store the agent can write to.
"""

In [ ]:
# Execute the skill above, using the shared runtime rather than a copy.
import glob, importlib.util, os, sys

# Kaggle mounts an attached kernel under /kaggle/input, and it uses two
# different layouts — /kaggle/input/<slug>/ on some kernels and
# /kaggle/input/notebooks/<user>/<slug>/ on others. Both were observed on the
# same account in the same hour, so match either. The recursive glob is cheap
# here because /kaggle/input holds only what is attached; globbing the working
# tree instead cost eleven seconds a notebook.
_WHERE = (sorted(glob.glob("/kaggle/input/**/cyber-commons-skill-runtime/__script__.py",
                           recursive=True))
          + [os.path.join(p, "skills/_runtime/cyber_commons_skill_runtime.py")
             for p in (".", "..", "../..")])
_found = next((p for p in _WHERE if os.path.isfile(p)), None)
if _found is None:
    # Say what was looked for and what is actually there. "The runtime is
    # missing" on its own costs whoever hits it an afternoon.
    raise SystemExit("The shared skill runtime is missing."
                     "  looked at: " + repr(_WHERE) +
                     "  /kaggle/input holds: " +
                     repr(glob.glob("/kaggle/input/**", recursive=True)[:20]) +
                     "  cwd: " + os.getcwd() +
                     ". On Kaggle it is attached to this notebook as a "
                     "source; locally it is skills/_runtime/ in the repository.")
_spec = importlib.util.spec_from_file_location("cyber_commons_skill_runtime", _found)
cyber_commons_skill_runtime = importlib.util.module_from_spec(_spec)
sys.modules["cyber_commons_skill_runtime"] = cyber_commons_skill_runtime
_spec.loader.exec_module(cyber_commons_skill_runtime)
from cyber_commons_skill_runtime import run_skill

# Split skills/secops/incident-scoping/SKILL.md into the two halves an agent uses —
# the frontmatter it routes on, and the body it follows.
meta, body = run_skill(SKILL_MD)

In [ ]:
# skills/secops/incident-scoping/scripts/incident_scoping.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Scope what an agent touched during an incident, from the run record rather than from the alert.

This is the executable half of the `incident-scoping` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

# The skill runtime comes from the shared library, not from a copy in this file.
# In a lesson notebook the cell above has already loaded it; standalone, find it
# the same way that cell does.
import glob as _glob, importlib.util as _ilu, os as _os, sys as _sys

if "cyber_commons_skill_runtime" not in _sys.modules:
    _where = (sorted(_glob.glob("/kaggle/input/**/cyber-commons-skill-runtime/__script__.py",
                                recursive=True))
              + [_os.path.join(p, "skills/_runtime/cyber_commons_skill_runtime.py")
                 for p in (".", "..", "../..",
                           _os.path.join(_os.path.dirname(__file__), "../../../_runtime"))])
    _found = next((p for p in _where if _os.path.isfile(p)), None)
    if _found is None:
        raise SystemExit("shared skill runtime not found; looked at " + repr(_where))
    _spec = _ilu.spec_from_file_location("cyber_commons_skill_runtime", _found)
    _mod = _ilu.module_from_spec(_spec)
    _sys.modules["cyber_commons_skill_runtime"] = _mod
    _spec.loader.exec_module(_mod)

from cyber_commons_skill_runtime import check, contract_of, parse_skill


def _skill_md():
    """The SKILL.md next to this script, or the one the notebook already parsed."""
    if "SKILL_MD" in globals():
        return globals()["SKILL_MD"]
    return (_pathlib.Path(__file__).resolve().parent.parent / "SKILL.md").read_text()


import pathlib as _pathlib

SKILL_MD = _skill_md()
meta, body = parse_skill(SKILL_MD)

REACHED = {
 "dana@corp":    ["repo-core", "repo-infra", "vault-dev"],
 "orchestrator": ["repo-core", "queue-tasks"],
 "patch-agent":  ["repo-core", "repo-payments"],
 "deploy-agent": ["cluster-prod"],
}
CHAIN = ["dana@corp", "orchestrator", "patch-agent", "deploy-agent"]

def scope(chain, reached):
    last_only = set(reached.get(chain[-1], []))
    full = {r for a in chain for r in reached.get(a, [])}
    return {"chain": " → ".join(chain),
            "scoped_last_actor_only": sorted(last_only),
            "scoped_whole_chain": sorted(full),
            "missed_by_naive_scoping": sorted(full - last_only),
            "undercount_factor": round(len(full)/len(last_only), 2) if last_only else None}

s = scope(CHAIN, REACHED)
for k, v in s.items(): print(f"{k:26s}{v}")
print("\nScoping the last actor finds one cluster. The chain reached six")
print("resources, including a payments repository and a dev vault.")

print(f"{'depth':>6}{'last-actor scope':>19}{'chain scope':>14}{'undercount':>12}")
print("-" * 52)
for d in range(1, 5):
    sub = CHAIN[:d]
    r = scope(sub, REACHED)
    print(f"{d:>6}{len(r['scoped_last_actor_only']):>19}"
          f"{len(r['scoped_whole_chain']):>14}"
          f"{str(r['undercount_factor']):>12}")
print("\nEach hop adds resources the last actor never touched. This is why B2.0")
print("bounds delegation depth: depth is an incident-scope multiplier.")

SHARED = {"repo-core": ["build-agent", "test-agent"],
          "cluster-prod": ["deploy-agent", "monitor-agent"],
          "repo-payments": ["finance-agent"]}

def scope_transitive(chain, reached, shared, hops=1):
    """Anything that shares a touched resource may have been influenced."""
    direct = {r for a in chain for r in reached.get(a, [])}
    exposed = set(chain)
    frontier = set(direct)
    for _ in range(hops):
        nxt = set()
        for res in frontier:
            for actor in shared.get(res, []):
                if actor not in exposed:
                    exposed.add(actor)
                    nxt |= set(reached.get(actor, []))
        frontier = nxt
    return {"resources_direct": sorted(direct),
            "actors_in_scope": sorted(exposed),
            "second_order_actors": sorted(exposed - set(chain))}

t = scope_transitive(CHAIN, REACHED, SHARED)
for k, v in t.items(): print(f"{k:22s}{v}")
print("\nFive more identities shared a resource with the compromised chain.")
print("They are not confirmed compromised — they are IN SCOPE, which is different")
print("and is the distinction an incident record has to make explicitly.")
assert t["second_order_actors"]

# Verify: produce the scope statement for the incident record.
def scope_statement(chain, reached, shared):
    s = scope(chain, reached)
    t = scope_transitive(chain, reached, shared)
    return (f"SCOPE\n"
            f"  chain              {s['chain']}\n"
            f"  confirmed touched  {s['scoped_whole_chain']}\n"
            f"  would have been missed by scoping the acting agent alone:\n"
            f"                     {s['missed_by_naive_scoping']}\n"
            f"  undercount factor  {s['undercount_factor']}×\n"
            f"  in scope, not confirmed (shared a resource):\n"
            f"                     {t['second_order_actors']}")
print(scope_statement(CHAIN, REACHED, SHARED))

contract = contract_of(body)
t = scope_transitive(CHAIN, REACHED, SHARED)
reach = sorted({r for a in CHAIN for r in REACHED.get(a, [])})

incident = {
 "window": {"first_suspicious_action": f"{CHAIN[1]} accepted an external instruction",
            "detected_at": "the deploy that followed",
            # the trigger precedes the detection by about one task loop
            "gap_seconds": 42 * 60},
 "chain": [{"action": f"{a} acted", "motivating_input": "issue comment"
                      if a == CHAIN[1] else f"instruction from {CHAIN[i]}",
            "input_origin": "external_untrusted" if a == CHAIN[1] else "internal",
            "within_authority": True}
           for i, a in enumerate(CHAIN[1:])],
 "root_cause": {"input": "issue comment on a public tracker",
                "origin": "external_untrusted",
                "why_trusted": "repository content was read as instruction, not data"},
 # every action was permitted; that is what makes this hard
 "authority": {"authorised_but_wrong": len(CHAIN) - 1, "exceeded_authority": 0},
 "data": {"reach": reach, "confirmed_exfiltration": [],
          "egress_bounded_by": "agent network policy"},
 "containment": {"cut": "credential",
                 "does_not_stop": sorted(t["second_order_actors"]),
                 "evidence_snapshotted_first": True},
 "clock": {"regulatory_trigger": False,
           "basis": "no confirmed exfiltration of personal data yet"},
}
problems = check(incident, contract)
print(f"conformance: {len(problems)} problem(s)")
for p in problems: print("   ", p)
assert not problems, problems

print(f"\nauthorised but wrong : {incident['authority']['authorised_but_wrong']}")
print(f"exceeded authority   : {incident['authority']['exceeded_authority']}")
print(f"reach                : {len(reach)} resources")
print(f"confirmed exfil      : {len(incident['data']['confirmed_exfiltration'])}")
print(f"revoking one credential does NOT stop: "
      f"{incident['containment']['does_not_stop'] or 'nothing else'}")
print()
print("Zero actions exceeded authority, and the incident still happened. That")
print("combination says the grant was too broad - a different fix from a")
print("control that failed, which is why the contract counts them separately.")
print()
print("Reach is 4 resources; confirmed exfiltration is 0. Reporting the second")
print("as the scope is how a notification decision gets made on the wrong number.")
assert incident["authority"]["exceeded_authority"] == 0
assert len(reach) > len(incident["data"]["confirmed_exfiltration"])

## What you just proved

Scoping the last actor finds `cluster-prod` alone; the whole chain reaches six resources, missing five, with an undercount factor of 6.0. The undercount grows with each hop. Transitive scoping adds five second-order identities that shared a resource, explicitly marked as in scope rather than confirmed compromised.

## Your turn

For your last incident involving a service account, recompute the scope by walking what else that account could reach. The number is almost always larger than what was written in the report.

---

**Next → [D2.4 · Containment at machine speed](https://spbreed.github.io/cyber-commons/lessons/D2.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D2.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D2.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*